In [2]:
import numpy as np

#Task 1 & 2

1.   **Implement the encryptor for a simplified
AES-like cipher**
2.   **Implement the decryptor for this simplified AES-like cipher**



### Parameter used for the task



*   p: prime number used;
*   n: number of encruption cycle;
*   u: plaintext message;



In [3]:
key = [1,0,0,0,0,0,0,0]
p = 11
n = 5
u = [1,0,0,0,0,0,0,0] # message

generating the different token used by AES starting from the key.

The generation map is taken from the instruction.

In [4]:
# Key token generated following the PDF schema
def get_token(key):
  key_token = [[key[0],key[2], key[4], key[6]], #1
              [key[0],key[1], key[2], key[3]], #2
              [key[0],key[3], key[4], key[7]], #3
              [key[0],key[3], key[5], key[6]], #4
              [key[0],key[2], key[5], key[7]], #5
              [key[2],key[3], key[4], key[5]]] #6
  return key_token
key_token = get_token(key)
print(f" token generated from the key {key}: \n{key_token}")

 token generated from the key [1, 0, 0, 0, 0, 0, 0, 0]: 
[[1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [0, 0, 0, 0]]


## Block function
The following function implements the different block of the AES chiper.

function **f(.)** used in the substitution block

In [5]:
#linear function used
def f(v, p, inverse = False):
  """Multiplies a number v by 2 and takes the result modulo p.
     If the invesre flag is set to True, the result is multiplied 2^(-1) module 11.
  Args:
    v: The number to multiply.
    p: The prime modulus.

  Returns:
    The result of 2 * v mod p.
  """
  a = 2
  if inverse:
    a = 6
  return (a * v) % p

In [6]:
def add_key(w, k, p, inverse = False):
  """Adds a key to a number.
     If the invesre flag is set to True, the result is subtracted.
  Args:
    w: The number to add the key to.
    k: The key to add.
    p: The prime modulus.

  Returns:
    The result of w + k mod p.
  """
  if inverse:
    return (w - np.concatenate((k, k))) % p
  else:
    return (w + np.concatenate((k, k))) % p

In [7]:
# substitution block
def substitution_block(v, p, inverse = False):
  """Applies the substitution block to the input u.
     The flag is passed to the function f.
  Args:
    u: The input to the substitution block.
    p: The prime modulus.

  Returns:
    The output of the substitution block.
  """
  return np.vectorize(f)(v, p, inverse)

In [8]:
# transposition block
def transposition_block(y, p):
  """Applies the transposition block to the input u.

  Args:
    u: The input to the transposition block.
    p: The prime modulus.

  Returns:
    The output of the transposition block.
  """
  return flat_matrix([y[0:4],y[7:3:-1]])

In [9]:
# linear block
def linear_block(z, p, inverse = False):
  """Applies the linear block to the input u.
     If inverse is set to True, the matrix is inverted before multiplication.
  Args:
    u: The input to the linear block.
    p: The prime modulus.

  Returns:
    The output of the linear block.
  """
  W = [[2, 5],[1, 7]]
  if inverse:
    W = [[2, 8],[6, 10]]
  w = np.dot(W, [z[0:4], z[4:]]) % p
  return flat_matrix(w)

In [10]:
def flat_matrix(w):
  """Flattens a matrix into a vector (used as standard between blocks).

  Args:
    w: The matrix to flatten.

  Returns:
    The flattened matrix.
  """
  return np.concatenate((w[0],w[1])).ravel()

## encryptor
We decided to store all of the passage of the encryptor, if only the encrypted message is needed it's possible to call the function encrypt

In [11]:
def encryptor(u, key_token, p, n):
  w = np.zeros((n+1, 8), dtype=int)
  w[0] = u

  for i in range(0, n):
    v = add_key(w[i], key_token[i], p)
    y = substitution_block(v, p)
    z = transposition_block(y, p)
    if(i < n - 1):
      w[i+1] = linear_block(z, p)
  w[n] = add_key(z, key_token[n], p)
  return w
w_en = encryptor(u, key_token, p, n)
print(w_en)

[[ 1  0  0  0  0  0  0  0]
 [ 8  0  0 10  4  0  0  3]
 [ 0  0  0  2  5  0  0  2]
 [ 2  0  0  2  8  0  0  0]
 [ 1  0  0 10  6  0  0  9]
 [ 4  0  0  9  7  0  0  3]]


## decryptor
We decided to store all of the passage of the decryptor, if only the clear message is needed it's possible to call the function decrypt

In [12]:
def decryptor(u, key_token, p, n):
  w = np.zeros((n+1, 8), dtype=int)
  w[n] = add_key(u, key_token[n], p, True)
  for i in range(n - 1, -1, -1):
    if(i < n-1):
      z = linear_block(w[i+1], p, True)
    else:
      z = w[n]
    y = transposition_block(z, p)
    v = substitution_block(y, p, True)
    w[i] = add_key(v, key_token[i], p, True)
  return w
w_dec = decryptor(w_en[n], key_token, p, n)
print(w_dec)

[[ 1  0  0  0  0  0  0  0]
 [ 8  0  0 10  4  0  0  3]
 [ 0  0  0  2  5  0  0  2]
 [ 2  0  0  2  8  0  0  0]
 [ 1  0  0 10  6  0  0  9]
 [ 4  0  0  9  7  0  0  3]]


## encrypt and decrypt

funtion used as wrapper for the encryptor / decryptor to only get the chiper / plain text.

In [13]:
def encrypt(u, key_token, p, n = 5):
  return encryptor(u, key_token, p, n)[n]
def decrypt(u, key_token, p, n = 5):
  return decryptor(u, key_token, p, n)[0]

## Testing

Using the plain text from the instruction **[1 0 0 0 0 0 0 0]**, the encrypted result should be **[4 0 0 9 7 0 0 3]**

In [14]:
print(f"encrypted message: {encrypt(u, key_token, p, n)}, should be [4 0 0 9 7 0 0 3]")
print(f"decrypted message: {decrypt(encrypt(u, key_token, p, n), key_token, p, n)}, shoul be [1 0 0 0 0 0 0 0]")

encrypted message: [4 0 0 9 7 0 0 3], should be [4 0 0 9 7 0 0 3]
decrypted message: [1 0 0 0 0 0 0 0], shoul be [1 0 0 0 0 0 0 0]


# Task 3

Identify the overall linear relationship for this simplified AES-like cipher

## Utility function

In [15]:
# generate vectors needed to identify linear relationship
def get_identity_matrix(size):
  """Generates an identity matrix of size size.

  Args:
    size: The size of the matrix.

  Returns:
    An identity matrix of size size.
  """
  return [[(1 if i == j else 0) for j in range(size)] for i in range(size)]
def get_empty_matrix(size):
  """Generates an empty matrix of size size.

  Args:
    size: The size of the matrix.

  Returns:
    An empty matrix of size size.
  """
  return [[0 for _ in range(size)] for _ in range(size)]
def get_identity_and_empty_matrix(size):
  """Generates an identity matrix and an empty matrix of size size.

  Args:
    size: The size of the matrix.

  Returns:
    A tuple containing the identity matrix and the empty matrix.
  """
  identity_matrix = get_identity_matrix(size)
  zero_matrix = get_empty_matrix(size)
  return identity_matrix, zero_matrix

In [16]:
def get_matrix(u, k, p):
  """Generates a matrix of size len(u) x len(x) where x is the encryption
      of the vector u[i] using the key k[i].

  Args:
    u: Matrix containing different plaintext.
    k: Matrix containing different keys.
    p: The prime modulus.

  Returns:
    A matrix of size len(u) x len(x).
  """
  out = []
  for i in range(len(u)):
    out.append(encrypt(u[i], get_token(k[i]), p))
  matrix = np.column_stack(out) # each vector obtained becomes a column
  return matrix

In [17]:
# calculate matrix A and B so that E(k, u) = ( Ak + Bu ) mod p
def get_A_B_matrixes(p, size):
  """Generates the matrix A and B of size size x size.
     Using the method described in the first appendix of the instructions.
  Args:
    p: The prime modulus.
    size: The size of the matrices.

  Returns:
    A tuple containing the matrices A and B.
  """
  identity, zeroes = get_identity_and_empty_matrix(size)
  A = get_matrix(zeroes, identity, p)
  B = get_matrix(identity, zeroes, p)
  return A, B

In [18]:
size = 8 # TO DO -> use u lenght
p = 11 # TO DO -> p is already decleared
A, B = get_A_B_matrixes(p, size)
print(A)
print(B)

[[ 9  0  1  6  0  0  1 10]
 [ 0  8  6  2  2  9  0  0]
 [ 0  6  0  8  3 10  0  0]
 [ 6  0  0  8  0  1  6  6]
 [ 2  0  1 10  0  0  1  3]
 [ 0  1  8  4  9  6  0  0]
 [ 0 10  0  5  7  6  0  0]
 [ 3  0  0  1  0  1  4  8]]
[[6 0 0 3 3 0 0 0]
 [0 6 3 0 0 3 0 0]
 [0 3 6 0 0 0 3 0]
 [3 0 0 6 0 0 0 3]
 [5 0 0 0 4 0 0 8]
 [0 5 0 0 0 4 8 0]
 [0 0 5 0 0 8 4 0]
 [0 0 0 5 8 0 0 4]]


## Testing

testing if the A, B matrix founded represent the correct linear relationship for the AES chyper

In [19]:
# test if matrixes work for encryption:
k = [1,0,0,0,0,0,0,0] # TO DO k, u are the same decleared before
u = [1,0,0,0,0,0,0,0]
p = 11

term1 = A @ k
term2 = B @ u
x = (term1 + term2) % p

print("Result:", x, ", should be [4 0 0 9 7 0 0 3]")

Result: [4 0 0 9 7 0 0 3] , should be [4 0 0 9 7 0 0 3]


# Task 4

From a known plaintext/ciphertext pair (u, x), implement a linear cryptanalysis KPA against
this cipher

## Modular algebra function

We im



In [20]:
# Computes the extended Euclidean Algorithm for (a, b)
# Returns (gcd, x, y) such that a*x + b*y = gcd
def extended_gcd(a, b):
  """Computes the extended Euclidean Algorithm for (a, b).

  Args:
    a: The first number.
    b: The second number.

  Returns:
    A tuple containing the gcd, x, and y such that a*x + b*y = gcd.
  """
  if a == 0:
      return b, 0, 1
  else:
      gcd, x1, y1 = extended_gcd(b % a, a)
      x = y1 - (b // a) * x1 # // is floor division
      y = x1
      return gcd, x, y

In [21]:
# Computes the modular multiplicative inverse of a modulo m
# Returns x such that (a * x) % m == 1
# Raises an exception if the inverse does not exist
def modInverse(a, m):
  """Computes the modular multiplicative inverse of a modulo m.

  Args:
    a: The number to find the inverse of.
    m: The modulus.

  Returns:
    The modular multiplicative inverse of a modulo m.
    Raises an exception if the inverse does not exist.
  """
  a = a % m
  gcd, x, y = extended_gcd(a, m)
  if gcd != 1:
      raise ValueError(f"Modular inverse does not exist for {a} modulo {m}")
  else:
      # x may be negative, so take it modulo m
      return x % m

In [22]:
# returns inverse of a matrix in mod p = ( A_tilde * det^-1(A) ) mod p
# where A_tilde is ( A^-1 * det(A) )
def inverse_matrix_mod_p(A, p):
  """Computes the inverse of a matrix in mod p.

  Args:
    A: The matrix to find the inverse of.
    p: The modulus.

  Returns:
    The inverse of A in mod p.
  """
  det_A = int(round(np.linalg.det(A)))
  A_inv = np.linalg.inv(A)
  A_tilde = A_inv * det_A
  inv_det_A = modInverse(det_A, p)
  A_inv_mod = np.round((A_tilde * inv_det_A)).astype(np.int64) % p
  return A_inv_mod.astype(int)

**Testing the inverse matrix function**

In [23]:
# Test inverse:
p = 11
A = np.array([[1, 2],
              [3, 4]])

A_inv = inverse_matrix_mod_p(A, p)
print("Result\n", A_inv)
print("Should be\n", "[[9 1]\n [7 5]]")

Result
 [[9 1]
 [7 5]]
Should be
 [[9 1]
 [7 5]]


## Linear KPA

implementig the linear cryptanalysis KPA

In [24]:
def find_linear_relationship(u, x, p):
  """Computes the linear relationship between u and x.

  Args:
    u: The plaintext.
    x: The ciphertext.
    p: The prime modulus.

  Returns:
    the k found from the linear relation Ak + Bu = x.
  """
  size = len(u)
  A, B = get_A_B_matrixes(p, size)
  A_inv = inverse_matrix_mod_p(A, p)
  return A_inv @ (x - B @ u) % p

In [25]:
def test_linear_attack(u, x, p):
  k = find_linear_relationship(u, x, p)
  print("----------------------------")
  print("found key: ", k)
  print("original chiper: ", x)
  print("recomputed chiper: ", encrypt(u, get_token(k), p))
  print("----------------------------")

verify the correctness of the function using the u, k previously decleared

In [26]:
u = [1,0,0,0,0,0,0,0] # TO DO -> x, u are already decleared
x = [4,0,0,9,7,0,0,3]
print(find_linear_relationship(u, x,p))

[1 0 0 0 0 0 0 0]


Carrying out the attack over the key/chiper pair givven

In [27]:
# from the test file:
KPA =[[[1,10,2,10,2,4,0,7],[0,5,4,2,2,6,1,3]],
     [[1,0,7,3,7,4,6,0],[5,4,0,5,10,4,6,2]],
     [[10,8,8,8,4,6,6,5],[10,6,8,0,6,8,5,1]],
     [[3,2,4,2,8,10,0,5],[6,3,3,9,9,1,4,3]],
     [[0,8,10,0,10,0,7,8],[10,5,1,8,4,3,4,10]]]
linear_key = np.zeros((len(KPA), 8), dtype=int)
for i in range(0, len(KPA)):
  linear_key[i] = find_linear_relationship(KPA[i][0],KPA[i][1], p)
  test_linear_attack(KPA[i][0], KPA[i][1], p)

----------------------------
found key:  [0 4 2 7 4 9 8 6]
original chiper:  [0, 5, 4, 2, 2, 6, 1, 3]
recomputed chiper:  [0 5 4 2 2 6 1 3]
----------------------------
----------------------------
found key:  [0 4 2 7 4 9 8 6]
original chiper:  [5, 4, 0, 5, 10, 4, 6, 2]
recomputed chiper:  [ 5  4  0  5 10  4  6  2]
----------------------------
----------------------------
found key:  [0 4 2 7 4 9 8 6]
original chiper:  [10, 6, 8, 0, 6, 8, 5, 1]
recomputed chiper:  [10  6  8  0  6  8  5  1]
----------------------------
----------------------------
found key:  [0 4 2 7 4 9 8 6]
original chiper:  [6, 3, 3, 9, 9, 1, 4, 3]
recomputed chiper:  [6 3 3 9 9 1 4 3]
----------------------------
----------------------------
found key:  [0 4 2 7 4 9 8 6]
original chiper:  [10, 5, 1, 8, 4, 3, 4, 10]
recomputed chiper:  [10  5  1  8  4  3  4 10]
----------------------------


summary of all the keys found

In [28]:
print(linear_key)

[[0 4 2 7 4 9 8 6]
 [0 4 2 7 4 9 8 6]
 [0 4 2 7 4 9 8 6]
 [0 4 2 7 4 9 8 6]
 [0 4 2 7 4 9 8 6]]


# Task 5
implement the encryptor for a simplified AES-like cipher with the parameters given in the
previous slides and the substitution function described by the following table

**the follwing code is redefining the substituion block following the instruction and then create the encryptor and decryptor that use the new substitution block**

In [29]:
def substitution_block_5(v, p, inverse = False):
  map = [[0,2,4,8,6,10,1,3,5,7,9],# forward map
           [0,6,1,7,2,8,4,9,3,10,5]]#inverse map
  x = 0
  if inverse:
    x = 1
  return np.vectorize(lambda x: map[int(inverse)][x])(v)

In [30]:
def encryptor_5(u, key_token, p, n):
  w = np.zeros((n+1, 8), dtype=int)
  w[0] = u

  for i in range(0, n):
    v = add_key(w[i], key_token[i], p)
    y = substitution_block_5(v, p)
    z = transposition_block(y, p)
    if(i < n - 1):
      w[i+1] = linear_block(z, p)
  w[n] = add_key(z, key_token[n], p)
  return w
w_en = encryptor_5(u, key_token, p, n)
print(w_en)

[[ 1  0  0  0  0  0  0  0]
 [ 8  0  0 10  4  0  0  3]
 [10  0  0  2  8  0  0  2]
 [ 9  0  0 10  6  0  0  9]
 [ 9  0  0  0  3  0  0  8]
 [ 9  0  0  0  5  0  0  6]]


In [31]:
def decryptor_5(u, key_token, p, n):
  w = np.zeros((n+1, 8), dtype=int)
  w[n] = add_key(u, key_token[n], p, True)
  for i in range(n - 1, -1, -1):
    if(i < n-1):
      z = linear_block(w[i+1], p, True)
    else:
      z = w[n]
    y = transposition_block(z, p)
    v = substitution_block_5(y, p, True)
    w[i] = add_key(v, key_token[i], p, True)
  return w
w_dec = decryptor_5(w_en[n], key_token, p, n)
print(w_dec)

[[ 1  0  0  0  0  0  0  0]
 [ 8  0  0 10  4  0  0  3]
 [10  0  0  2  8  0  0  2]
 [ 9  0  0 10  6  0  0  9]
 [ 9  0  0  0  3  0  0  8]
 [ 9  0  0  0  5  0  0  6]]


In [32]:
def encrypt_5(u, key_token, p, n = 5):
  return encryptor_5(u, key_token, p, n)[n]
def decrypt_5(u, key_token, p, n = 5):
  return decryptor_5(u, key_token, p, n)[0]

# Task 6

Find a linear approximation of the cipher in Task 5

evaluate A, B using the new encriptor and then evaluate C such that Ak + Bu +Cx -> 0

rewriting the function from the previous task using the new encryption function

The idea here is to consider C as the Identity matrix, then find some possible key from the linear system Ak + Bu + Cx = 0 for each KPA_pair and then try key near the finded one in the key space to see if we can restrict the searching space and compute the key.

In [33]:
# calculate matrix A or B so that E(k, u) = ( Ak + Bu ) mod p
def get_matrix_5(u, k, p):
  out = []
  for i in range(len(u)):
    out.append(encrypt_5(u[i], get_token(k[i]), p))
  matrix = np.column_stack(out) # each vector obtained becomes a column
  return matrix


def get_decrypt_matrix(B, u, k, p):
  out = []

  for i in range(len(u)):
    print(u[i])
    out.append(B @ decrypt_5(u[i], get_token(k[i]), p))
    print(B @ decrypt_5(u[i], get_token(k[i]), p))
  matrix = np.column_stack(out)
  return matrix

In [34]:
# calculate matrix A and B so that E(k, u) = ( Ak + Bu ) mod p
def get_A_B_C_matrixes_5(p, size):
  identity, zeros = get_identity_and_empty_matrix(size)
  A = get_matrix_5(zeros, identity, p)
  B = get_matrix_5(identity, zeros, p)
  C = get_identity_matrix(size)#-get_decrypt_matrix(B, identity, zeros, p)
  return A, B, C

## Testing

In [35]:
KPA_pair = [[[8,4,5,7,10,3,9,8],[10,1,2,7,8,9,8,2]],
           [[0,10,7,2,4,1,2,2],[8,1,2,2,4,5,8,6]],
           [[4,2,9,8,9,3,7,7],[8,7,2,2,4,5,7,8]],
           [[5,6,9,5,2,4,4,10],[7,5,3,1,1,2,1,3]],
           [[7,4,9,8,3,4,4,8],[0,5,5,4,0,9,10,10]]]

In [84]:
A, B, C = get_A_B_C_matrixes_5(p, 8)
print(A)
print(B)
print(C)

[[ 4  0  1  4  0  0 10 10]
 [ 0  3  1  0  9  9  0  0]
 [ 0  4  3  8 10 10  0  0]
 [ 4  0  0  3  0  1  2  7]
 [ 8  0  1  8  0  0  9  5]
 [ 0  7  2 10  8  8  0  0]
 [ 0  8  9  0  2  8  0  0]
 [ 3  0  0  7  0  1  4  6]]
[[9 0 0 2 6 0 0 2]
 [0 9 2 0 0 6 2 0]
 [0 2 9 0 0 2 6 0]
 [2 0 0 9 2 0 0 6]
 [9 0 0 5 2 0 0 2]
 [0 9 5 0 0 2 2 0]
 [0 5 9 0 0 2 2 0]
 [5 0 0 9 2 0 0 2]]
[[1, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 1]]


find the key that can solve the system with the message and the chiper passed

In [85]:
###find key and test similar one
def find_linear_relationship_2(A,B,C,u, x, p):
  """Computes the linear relationship between u and x.

  Args:
    u: The plaintext.
    x: The ciphertext.
    p: The prime modulus.

  Returns:
    the k found from the linear relation Ak + Bu = x.
  """
  size = len(u)
  A_inv = inverse_matrix_mod_p(A, p)
  u = np.array(u)
  x = np.array(x)
  return A_inv @ (C @ x - B @ u) % p

map all the key obtained from the linear relationship and the KPA pair into a map, where the encryption key is the map key and the value is the number of times the key is returned from the relationship

In [38]:
# prompt: for each KPA_pair call find_linear_relationship and store the result in a map where the result is the key and the number of time it appears is the value

from collections import defaultdict

results_map = defaultdict(int)

for kpa_pair in KPA_pair:
  u = kpa_pair[0]
  x = kpa_pair[1]
  key = find_linear_relationship_2(A, B, C, u, x, p)
  results_map[tuple(key)] +=1

print(results_map)


[[ 4  0  1  4  0  0 10 10]
 [ 0  3  1  0  9  9  0  0]
 [ 0  4  3  8 10 10  0  0]
 [ 4  0  0  3  0  1  2  7]
 [ 8  0  1  8  0  0  9  5]
 [ 0  7  2 10  8  8  0  0]
 [ 0  8  9  0  2  8  0  0]
 [ 3  0  0  7  0  1  4  6]]
[[ 4  0  1  4  0  0 10 10]
 [ 0  3  1  0  9  9  0  0]
 [ 0  4  3  8 10 10  0  0]
 [ 4  0  0  3  0  1  2  7]
 [ 8  0  1  8  0  0  9  5]
 [ 0  7  2 10  8  8  0  0]
 [ 0  8  9  0  2  8  0  0]
 [ 3  0  0  7  0  1  4  6]]
[[ 4  0  1  4  0  0 10 10]
 [ 0  3  1  0  9  9  0  0]
 [ 0  4  3  8 10 10  0  0]
 [ 4  0  0  3  0  1  2  7]
 [ 8  0  1  8  0  0  9  5]
 [ 0  7  2 10  8  8  0  0]
 [ 0  8  9  0  2  8  0  0]
 [ 3  0  0  7  0  1  4  6]]
[[ 4  0  1  4  0  0 10 10]
 [ 0  3  1  0  9  9  0  0]
 [ 0  4  3  8 10 10  0  0]
 [ 4  0  0  3  0  1  2  7]
 [ 8  0  1  8  0  0  9  5]
 [ 0  7  2 10  8  8  0  0]
 [ 0  8  9  0  2  8  0  0]
 [ 3  0  0  7  0  1  4  6]]
[[ 4  0  1  4  0  0 10 10]
 [ 0  3  1  0  9  9  0  0]
 [ 0  4  3  8 10 10  0  0]
 [ 4  0  0  3  0  1  2  7]
 [ 8  0  1  8  0  0  9  

In [88]:
# prompt: generate a code that print different key from the one passed as parameter that are similar to the one passed

import numpy as np
from collections import defaultdict

def find_similar_keys(key, key_number):
  similar_keys = []
  for i in range(0, key_number):
    random_key = (np.random.randint(-3, 3, size=8))
    similar_keys.append((key - random_key)%p)
  return similar_keys

# Example usage (assuming results_map and key are defined as in your original code)
# Replace this with your actual key and results_map


#print(f"Keys similar to {key}: {similar_keys_list}")

def test_key(key_list, KPA, p):
  for j in range(len(key_list)):
    for i in range(0, len(KPA)):
      correct = 0
      if  np.array_equal(encrypt_5(KPA[i][0], get_token(key_list[j]), p), KPA[i][1]):
        correct += 1
        print("halo")
      if(correct == len(KPA)):
        print(f"Key {key_list[j]} is correct")


for key, value in results_map.items():
  print(f"Key: {key}")
  similar_keys_list = find_similar_keys(key, 6**6)
  test_key(similar_keys_list, KPA_pair, p)


Key: (np.int64(6), np.int64(9), np.int64(3), np.int64(6), np.int64(5), np.int64(1), np.int64(4), np.int64(1))
Key: (np.int64(0), np.int64(4), np.int64(5), np.int64(2), np.int64(8), np.int64(2), np.int64(10), np.int64(5))
Key: (np.int64(8), np.int64(0), np.int64(2), np.int64(4), np.int64(0), np.int64(4), np.int64(0), np.int64(8))
Key: (np.int64(0), np.int64(1), np.int64(2), np.int64(1), np.int64(3), np.int64(5), np.int64(7), np.int64(2))
Key: (np.int64(0), np.int64(2), np.int64(10), np.int64(4), np.int64(3), np.int64(7), np.int64(10), np.int64(8))


# Task 7

implement the encryptor for a simplified AES-like cipher using a new key generation rule and a substitution block.

In [43]:
key = [1,0,0,0]
u = [1,0,0,0,0,0,0,0]
# Key token generated following the PDF schema for the task 8
def get_token_2(key):
  """Generates the token for the given key.

  Args:
    key: The key to generate the token for.

  Returns:
    The token for the given key.
  """
  key_token = [[key[0],key[1], key[2], key[3]], #1
              [key[0],key[1], key[3], key[2]], #2
              [key[1],key[2], key[3], key[0]], #3
              [key[0],key[3], key[1], key[2]], #4
              [key[2],key[3], key[0], key[1]], #5
              [key[1],key[3], key[0], key[2]]] #6
  return key_token

# Generate the new token
key_token_2 = get_token_2(key)

# Defining the
def mod_inverse(a, p):
  """Computes the modular multiplicative inverse of a modulo p.

  Args:
    a: The number to find the inverse of.
    p: The modulus.

  Returns:
    The modular multiplicative inverse of a modulo p.
    Raises an exception if the inverse does not exist.
  """
  #computes the multiplicative inverse in the GF using Fermat's theorem
  return pow(int(a), int(p-2), int(p))



def f_non_linear(v, p, inverse = False):
    """Multiplies a number v by 2 and takes the result modulo p.
        If the invesre flag is set to True, the result is multiplied 2^(-1) module 11.
    Args:
        v: The number to multiply.
        p: The prime modulus.

    Returns:
        The result of 2 * v mod p.
    """
    a = 2
    if inverse:
        a = 6
        return (mod_inverse(a*v, p)) % p

    return (a * mod_inverse(v, p)) % p


def non_linear_substitution_block(v, p, inverse = False):
  """Applies the non-linear substitution block to a vector v.

  Args:
    v: The vector to apply the non-linear substitution block to.
    p: The prime modulus.
  """
  return np.vectorize(f_non_linear)(v, p, inverse)

Redefining the encryptor for the task 7 to implement the new substitution block

In [45]:
def encryptor_7(u, key_token, p, n):
  """Encrypts a plaintext u using the given key token.

  Args:
    u: The plaintext to encrypt.
    key_token: The key token to use for encryption.
    p: The prime modulus.
  """
  w = np.zeros((n+1, 8), dtype=int)
  w[0] = u

  for i in range(0, n):
      v = add_key(w[i], key_token[i], p)
      y = non_linear_substitution_block(v, p)
      z = transposition_block(y, p)
      if(i < n - 1):
          w[i+1] = linear_block(z, p)

  w[n] = add_key(z, key_token[n], p)
  return w


def encrypt_7(u, key_token, p, n = 5):
    return encryptor_7(u, key_token, p, n)[n]

In [46]:
def decryptor_7(u, key_token, p, n):
  """Decrypts a ciphertext u using the given key token.

  Args:
    u: The ciphertext to decrypt.
    key_token: The key token to use for decryption.
    p: The prime modulus.
  """
  w = np.zeros((n+1, 8), dtype=int)
  w[n] = add_key(u, key_token[n], p, True)
  for i in range(n - 1, -1, -1):
    if(i < n-1):
      z = linear_block(w[i+1], p, True)
    else:
      z = w[n]
    y = transposition_block(z, p)
    v = non_linear_substitution_block(y, p, True)
    w[i] = add_key(v, key_token[i], p, True)
  return w


def decrypt_7(u, key_token, p, n = 5):
  return decryptor_7(u, key_token, p, n)[0]

#Task 8

Implement a “meet-in-the-middle” attack see Appendix 3 against the concatenation of two
instances of the non linear simplified AES-like cipher defined in Task 7, with different keys
k′, k′′, respectively.

KPA test: declering the key givven as assignment

In [47]:
KPA_pair = [[[6,5,0,2,9,0,9,0],[1,7,5,9,6,3,4,9]],
           [[7,5,2,6,1,7,6,0],[0,0,4,7,3,0,7,8]],
           [[0,1,0,4,9,6,5,9],[5,0,1,1,9,0,5,10]],
           [[1,9,1,5,1,6,0,8],[8,8,1,10,7,2,0,3]],
           [[9,10,10,5,2,1,5,6],[0,3,8,5,10,1,5,2]]]

**Since the key are 11^4** we decided to not generate them random but sequencially generate them

In [48]:
def generate_key(i, p=11, k_size = 4):
  """Generates a key of length k_size using the given index i.

  Args:
    i: The index to use for generating the key.
  """
  k = np.zeros(k_size, dtype=int)
  for n in range(k_size - 1, -1, -1):
    if i > p**n:
      k[k_size - 1 - n] = math.floor(i / p**n)
      i = i - k[k_size - 1 - n] * p**n
  return k
#returns an array of 4 elements that reperesent the base11 version of the number n (max value 14640)
#this is used to test all the possible keys of the exercise, since they are just 11^4
def base11(n):
  """ Function used at the end """
  if n > 14640:
     return -1

  base11_representation = []
  while n > 0:
    base11_representation.insert(0, n % 11)
    n //= 11

  while len(base11_representation) < 4:
    base11_representation.insert(0, 0)

  return base11_representation[:4]

### Testing the correctness of the algorithm
base11(11121)

[8, 3, 10, 0]

Generate two table with "num_guesses" entry for a givven pair of chiper and plain text, in particular the first table contains the encryption of the plain text while the second one contains the decryption of the chiper.

In [49]:
def generate_guesses(num_guesses, KPA_pair):
  table1 = []

  table2 = []

  #define on which pair to test
  PT = KPA_pair[0]
  CT = KPA_pair[1]
  #create the first set of keys and compute ciphertext
  #then add them to the table
  for i in range(num_guesses):
    key = base11(i)       #but we exhaust all possible keys
    key_token = get_token_2(key)
    x = encrypt_7(PT, key_token, 11)
    table1.append([tuple(x), key])

  table1 = sorted(table1, key=lambda x: x[0])

  #create the second set of keys and compute plaintext
  #then add them to the table
  for j in range(num_guesses):
    key = base11(j)       #but we exhaust all possible keys
    key_token = get_token_2(key)
    u = decrypt_7(CT, key_token, 11)
    table2.append([tuple(u), key])

  #since we saved the text as a tuple, it follows a total ordering and it's possible to sort the table
  table2 = sorted(table2, key=lambda x: x[0])
  return table1, table2

The code look for the common element between two table to find a matches betwen the encrypted plain text and the decrypted chiper one.

In [50]:
#find all common elements in the 2 tables
def common_elements(table1, table2):
  """Finds the common elements between two tables based on the first columns of
     the table.

  Args:
    table1: The first table.
    table2: The second table.
  """
  n1 = len(table1)
  n2 = len(table2)
  common_elements = []
  i = 0
  j = 0
  while i < n1 and j < n2:
      if table1[i][0] == table2[j][0]:
          common_elements.append((table1[i], table2[j]))
          i += 1
          j += 1
      elif table1[i][0] < table2[j][0]:
          i += 1
      else:
          j += 1
  return common_elements
  #print("Common elements:", common_elements)


For each KPA_pair, the algorithm will find the possible key that will encrypt the plain text in the final chiper text using the "common_elements" function. Then it will build a map where the keys are the possible keys that works for a KPA-pair, and as value the number of KPA_pair that are solved by the specific key.

In [51]:
num_guesses = 11**4
key_list = {}
for i in range(len(KPA_pair)):
  table1, table2 = generate_guesses(num_guesses, KPA_pair[i])
  common_elements_list = common_elements(table1, table2)
  for r in common_elements_list:
    k1 = r[0][1]
    k2 = r[1][1]
    tuple_key = tuple((tuple(k1), tuple(k2)))
    if tuple_key in key_list:
      key_list[tuple_key] += 1
    else:
      key_list[tuple_key] = 1

Look in the previous map for a key that has a value equal to the number of KPA_pair and so for a key that works for evry pair.

In [53]:
for key, value in key_list.items():
    k1, k2 = key
    if(value == len(KPA_pair)):
      print(f"k1: {k1}, k2: {k2}, Count: {value}")

k1: (8, 4, 7, 10), k2: (8, 7, 1, 4), Count: 5
